In [ ]:
# ============================================================
# CELL 0: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip -q install zstandard orjson tqdm


In [ ]:
# ============================================================
# CELL 2: Copy raw data to local SSD for fast I/O
# ============================================================
# Google Drive FUSE mount chỉ đạt 10-50 MB/s, trong khi local SSD
# đạt 500+ MB/s. Copy file zst về local trước khi xử lý tiết kiệm
# hàng giờ I/O time.
import os, time, shutil

BASE_DIR = '/content/drive/MyDrive/chess_engine/data'
RAW_DIR_DRIVE = '/content/drive/MyDrive/chess_engine/data/raw/lichess_db_eval.jsonl.zst'
PROC_DIR_DRIVE = os.path.join(BASE_DIR, 'process')

# Local paths (SSD)
LOCAL_DATA_DIR = '/content/chess_data'
RAW_DIR_LOCAL = os.path.join(LOCAL_DATA_DIR, 'lichess_db_eval.jsonl.zst')
PROC_DIR_LOCAL = os.path.join(LOCAL_DATA_DIR, 'process')

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
os.makedirs(PROC_DIR_LOCAL, exist_ok=True)
os.makedirs(PROC_DIR_DRIVE, exist_ok=True)

# Copy zst to local if not already present
if not os.path.exists(RAW_DIR_LOCAL):
    print(f"Copying {RAW_DIR_DRIVE} -> {RAW_DIR_LOCAL} ...")
    t0 = time.time()
    shutil.copy2(RAW_DIR_DRIVE, RAW_DIR_LOCAL)
    dt = time.time() - t0
    size_gb = os.path.getsize(RAW_DIR_LOCAL) / (1 << 30)
    print(f"Copied {size_gb:.2f} GB in {dt:.1f}s ({size_gb/dt*1024:.1f} MB/s)")
else:
    size_gb = os.path.getsize(RAW_DIR_LOCAL) / (1 << 30)
    print(f"Local copy already exists: {RAW_DIR_LOCAL} ({size_gb:.2f} GB)")

# Use local path for processing
RAW_DIR = RAW_DIR_LOCAL
PROC_DIR = PROC_DIR_LOCAL
print(f"\nRAW_DIR: {RAW_DIR}")
print(f"PROC_DIR: {PROC_DIR}")


In [ ]:
# ============================================================
# CELL 3: Imports + CONFIG (CRANE-v0 format)
# ============================================================
import os, time, math, zlib, hashlib, json
import numpy as np
import zstandard as zstd
import orjson
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing

import sys
sys.path.insert(0, '/content/drive/MyDrive/chess_engine/')

from core.board import Board
from representation.encode import encode_crane_v0

# ================== ENCODE SCHEMA ==================
ENCODE_SCHEMA = "crane_v0_stm_spatial18_scalar5"
N_SPATIAL_CHANNELS = 18
N_SCALAR_DIMS = 5

# ================== EVAL FILTERS ==================
CP_SCALE = 600.0
FIXED_DEPTH = 25
MIN_KNODES = 50_000
DEPTH_POLICY = "at_least"   # "exact" hoặc "at_least"

# POV config:
SOURCE_CP_POV = "white"   # lichess_db_eval là white POV
LABEL_POV = "stm"         # encode_board là STM-relative

# ================== OPENING FILTER ==================
MAX_PIECES_OPENING = 28
PHASE_OPENING_MIN = 20
PHASE_W = {'N': 1, 'B': 1, 'R': 2, 'Q': 4}

# ================== MATE / EXTREME FILTER ==================
MIN_MATE_DISTANCE = 3
MAX_ABS_CP = 1200
KEEP_MATE_PROB = 0.10
SAMPLE_PROB = 1.0

# ================== DIRTY SAMPLE FILTER ==================
DIRTY_Y_THRESHOLD = 0.05
DIRTY_MAT_THRESHOLD = 2.0

# ================== TARGET OUTPUT ==================
TARGET_TOTAL = 5_000_000
SHARD_SIZE = 50_000
SPLIT_RATIO = {"train": 0.8, "val": 0.1, "test": 0.1}

# ================== BUCKET BALANCING ==================
N_BUCKETS = 20
BUCKET_EDGES = np.linspace(-1.0, 1.0, N_BUCKETS + 1)

TARGET_BUCKET_WEIGHTS = np.array([
    0.014, 0.022, 0.034, 0.035, 0.029,
    0.027, 0.029, 0.035, 0.060, 0.215,
    0.215, 0.060, 0.035, 0.029, 0.027,
    0.029, 0.035, 0.034, 0.022, 0.014
], dtype=np.float64)
TARGET_BUCKET_WEIGHTS /= TARGET_BUCKET_WEIGHTS.sum()

# Bucket balancing chỉ cho train. Val/test giữ phân phối tự nhiên để đánh giá đúng.
APPLY_BUCKET_QUOTA = {"train": True, "val": False, "test": False}

# ================== PERFORMANCE ==================
# ProcessPoolExecutor chỉ hoạt động ổn định với 'fork' start method (Linux/Colab).
# Trên macOS/Windows (spawn), worker không thể pickle notebook functions.
if multiprocessing.get_start_method() == "fork":
    N_WORKERS = max(1, os.cpu_count() - 1)
else:
    N_WORKERS = 1
    print("WARNING: Non-fork start method detected. Parallel encoding disabled.")

# ================== VERIFICATION ==================
VERIFY_FIRST_N = 200   # Verify encoding cho N sample đầu tiên mỗi split

# ================== TRY IMPORT verify_encoding ==================
try:
    from representation.encode import verify_encoding as _verify_encoding_fn
except ImportError:
    _verify_encoding_fn = None
    print("NOTE: verify_encoding not found in representation.encode. "
          "Encode verification will use structural checks only.")

print(f"ENCODE_SCHEMA: {ENCODE_SCHEMA}")
print(f"N_SPATIAL_CHANNELS: {N_SPATIAL_CHANNELS}, N_SCALAR_DIMS: {N_SCALAR_DIMS}")
print(f"N_WORKERS: {N_WORKERS}")
print(f"VERIFY_FIRST_N: {VERIFY_FIRST_N}")


In [ ]:
# ============================================================
# CELL 3.5: Encode import safety check
# ============================================================
# Verify that the imported encode_crane_v0 has correct flip convention.
# This catches the case where the GitHub repo still has the old
# encode.py with INVERTED flip logic (Row 7 = STM instead of Row 0 = STM).

print("Checking encode_crane_v0 flip convention...")

# Test: White starting position → White King must be on row 0
_test_fen_w = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
_test_board_w = Board(_test_fen_w)
_test_X_w, _test_s_w = encode_crane_v0(_test_board_w)

_w_king_rows = np.where(_test_X_w[5].sum(axis=1) > 0.5)[0]
_b_king_rows = np.where(_test_X_w[11].sum(axis=1) > 0.5)[0]

if len(_w_king_rows) != 1 or _w_king_rows[0] != 0:
    raise RuntimeError(
        f"FLIP BUG DETECTED: White STM King at row {_w_king_rows}, expected row 0. "
        f"The encode_crane_v0 function has inverted flip logic! "
        f"Push the fixed encode.py to GitHub before running this notebook."
    )

# Test: Black STM → Black King must be on row 0 (after flip)
_test_fen_b = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"
_test_board_b = Board(_test_fen_b)
_test_X_b, _test_s_b = encode_crane_v0(_test_board_b)

_b2_king_rows = np.where(_test_X_b[5].sum(axis=1) > 0.5)[0]
if len(_b2_king_rows) != 1 or _b2_king_rows[0] != 0:
    raise RuntimeError(
        f"FLIP BUG DETECTED: Black STM King at row {_b2_king_rows}, expected row 0. "
        f"The encode_crane_v0 function has inverted flip logic!"
    )

# Test: Output shape
if _test_X_w.shape != (18, 8, 8) or _test_s_w.shape != (5,):
    raise RuntimeError(
        f"Shape mismatch: X={_test_X_w.shape}, s={_test_s_w.shape}. "
        f"Expected X=(18,8,8), s=(5,). Check encode schema version."
    )

print("✅ encode_crane_v0 flip convention: CORRECT (Row 0 = STM back rank)")
print("✅ Output shape: CORRECT (18×8×8 spatial + 5-dim scalar)")
print("✅ Import safety check: PASSED")


In [ ]:
# ============================================================
# CELL 4: Utility functions (updated for CRANE-v0)
# ============================================================

def ensure_full_fen(fen: str) -> str:
    """Đảm bảo FEN có đủ 6 trường (thêm halfmove=0, fullmove=1 nếu thiếu)."""
    parts = fen.split()
    return (fen + " 0 1") if len(parts) == 4 else fen


def canonical_fen_for_split(fen: str) -> str:
    """Canonical key cho split — bỏ halfmove/fullmove để tránh leakage."""
    parts = fen.split()
    return " ".join(parts[:4]) if len(parts) >= 4 else fen.strip()


def split_by_fen(fen: str, train=0.8, val=0.1) -> str:
    """Deterministic split dựa trên CRC32 của canonical FEN."""
    key = canonical_fen_for_split(fen)
    h = zlib.crc32(key.encode("utf-8")) & 0xffffffff
    r = h / 2**32
    if r < train:
        return "train"
    elif r < train + val:
        return "val"
    return "test"


def pick_eval_by_depth(evals, fixed_depth: int, min_knodes: int, policy: str):
    """Chọn evaluation tốt nhất theo depth/knodes policy."""
    cands = []
    for e in evals:
        depth = int(e.get("depth", -1))
        if policy == "exact":
            if depth != fixed_depth:
                continue
        elif policy == "at_least":
            if depth < fixed_depth:
                continue
        else:
            raise ValueError(f"Unsupported DEPTH_POLICY: {policy}")

        if int(e.get("knodes", 0)) < min_knodes:
            continue

        pvs = e.get("pvs")
        if not pvs:
            continue

        cands.append(e)

    if not cands:
        return None

    return max(cands, key=lambda x: (int(x.get("depth", -1)), int(x.get("knodes", -1))))


def mate_to_cp(mate: int) -> float:
    """Chuyển mate score sang centipawn tương đương."""
    sign = 1.0 if mate > 0 else -1.0
    m = min(abs(int(mate)), 100)
    return sign * (10000.0 - 100.0 * m)


def pv_passes_mate_and_extreme_filters(pv) -> bool:
    """Kiểm tra PV có hợp lệ không (mate distance và cp range)."""
    if "mate" in pv:
        return abs(int(pv["mate"])) >= MIN_MATE_DISTANCE
    if "cp" in pv:
        return abs(int(pv["cp"])) <= MAX_ABS_CP
    # PV không có cả cp lẫn mate → drop
    return False


def fen_board_piece_count_and_phase(board_part: str):
    """Đếm số quân và tính phase từ phần board của FEN."""
    pieces = 0
    phase = 0
    for ch in board_part:
        if ch == '/' or ch.isdigit():
            continue
        pieces += 1
        phase += PHASE_W.get(ch.upper(), 0)
    return pieces, phase


def is_opening_fen(fen: str) -> bool:
    """Kiểm tra FEN có thuộc giai đoạn opening không."""
    board_part = fen.split()[0]
    pieces, phase = fen_board_piece_count_and_phase(board_part)
    return (pieces > MAX_PIECES_OPENING) and (phase >= PHASE_OPENING_MIN)


def cp_from_source_to_white(cp: float, stm: str, source_pov: str) -> float:
    if source_pov == "white":
        return cp
    if source_pov == "stm":
        return cp if stm == "w" else -cp
    raise ValueError(f"Unsupported SOURCE_CP_POV: {source_pov}")


def cp_from_white_to_label(cp_white: float, stm: str, label_pov: str) -> float:
    if label_pov == "white":
        return cp_white
    if label_pov == "stm":
        return cp_white if stm == "w" else -cp_white
    raise ValueError(f"Unsupported LABEL_POV: {label_pov}")


def pv_to_label_value(pv, stm: str) -> float:
    """Chuyển PV thành label value y ∈ (-1, 1).

    Clip y ∈ [-0.9999, 0.9999] để tránh atanh(±1) = ±∞
    khi training tính z* = atanh(clip(y, -0.999, 0.999)).
    Mate scores tạo tanh(x) ≈ 1.0 trong float32 → phải clip.
    """
    if "cp" in pv:
        cp_src = float(pv["cp"])
    else:
        cp_src = mate_to_cp(int(pv["mate"]))

    cp_white = cp_from_source_to_white(cp_src, stm=stm, source_pov=SOURCE_CP_POV)
    cp_label = cp_from_white_to_label(cp_white, stm=stm, label_pov=LABEL_POV)
    y = math.tanh(cp_label / CP_SCALE)
    # Defense-in-depth: đảm bảo y không bao giờ bằng chính xác ±1.0
    return max(-0.9999, min(0.9999, y))


def bucket_id(y: float, edges: np.ndarray) -> int:
    """Xác định bucket index cho giá trị y."""
    i = int(np.searchsorted(edges, y, side="right") - 1)
    return max(0, min(i, len(edges) - 2))


def build_bucket_quota(total: int, weights: np.ndarray) -> np.ndarray:
    """Phân bổ quota cho từng bucket."""
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    raw = total * w
    q = np.floor(raw).astype(np.int64)
    rem = int(total - q.sum())
    if rem > 0:
        frac = raw - q
        idx = np.argsort(-frac)[:rem]
        q[idx] += 1
    return q


def shard_caps_from_total(total: int, shard_size: int) -> np.ndarray:
    """Tính capacity cho từng shard."""
    if total <= 0:
        return np.zeros((0,), dtype=np.int64)
    n = int((total + shard_size - 1) // shard_size)
    caps = np.full((n,), int(shard_size), dtype=np.int64)
    caps[-1] = int(total - shard_size * (n - 1))
    return caps


def compute_raw_material_delta(fen: str) -> float:
    """Tính material delta (STM perspective) từ FEN string."""
    piece_values = {
        'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9, 'K': 0,
        'p': 1, 'n': 3, 'b': 3, 'r': 5, 'q': 9, 'k': 0
    }
    parts = fen.split()
    board_part = parts[0]
    stm = parts[1] if len(parts) > 1 else "w"

    white_mat = black_mat = 0.0
    for ch in board_part:
        if ch in piece_values:
            if ch.isupper():
                white_mat += piece_values[ch]
            else:
                black_mat += piece_values[ch]

    return (white_mat - black_mat) if stm == "w" else (black_mat - white_mat)


def is_dirty_sample(y: float, fen: str) -> bool:
    """Phát hiện dirty samples: |y| ≈ 0 nhưng material imbalance lớn."""
    if abs(y) >= DIRTY_Y_THRESHOLD:
        return False
    return abs(compute_raw_material_delta(fen)) > DIRTY_MAT_THRESHOLD


def sha256_file(path: str) -> str:
    """Tính SHA256 hash của file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(1 << 20)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def save_hist_png(y_values, edges, out_path, title):
    """Lưu histogram của y values."""
    plt.figure(figsize=(8, 4))
    plt.hist(y_values, bins=edges, edgecolor='white', linewidth=0.5)
    plt.title(title)
    plt.xlabel("y")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


# Worker function cho parallel encoding (phải ở module level để pickle)
# NOTE: Chỉ hoạt động với 'fork' start method (Linux/Colab).
# Với 'spawn' (macOS/Windows), sẽ fallback về sequential encoding.
def _encode_fen_worker(fen: str):
    """Encode một FEN → (X_float16, s_float16).
    
    Returns None if encode fails (invalid FEN, bad en passant, etc.)
    """
    try:
        b = Board(ensure_full_fen(fen))
        X, s = encode_crane_v0(b)
        return X.astype(np.float16), s.astype(np.float16)
    except Exception:
        return None


print("Utility functions loaded.")


In [ ]:
# ============================================================
# CELL 5: Encode pre-flight verification
# ============================================================
# Verify rằng encode_crane_v0 hoạt động đúng trước khi chạy pipeline.
# Điều này bắt các lỗi như: import sai, flip convention sai,
# shape không khớp CRANE spec, v.v.

print("=" * 60)
print("ENCODE PRE-FLIGHT VERIFICATION")
print("=" * 60)

# Test 1: Basic encode
test_fen = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"
b = Board(test_fen)
X, s = encode_crane_v0(b)

print(f"\n[1] Basic encode test:")
print(f"    FEN: {test_fen}")
print(f"    X shape: {X.shape}  (expected: (18, 8, 8))")
print(f"    s shape: {s.shape}  (expected: (5,))")
print(f"    X dtype: {X.dtype}  s dtype: {s.dtype}")
assert X.shape == (18, 8, 8), f"X shape mismatch: {X.shape}"
assert s.shape == (5,), f"s shape mismatch: {s.shape}"

# Test 2: STM King exists (Row 0 = STM convention — orientation, not position constraint)
stm_king_plane = 5  # Plane 5 = K_stm
king_rows = np.where(X[stm_king_plane].sum(axis=1) > 0.5)[0]
print(f"\n[2] STM King position (Black STM → flip):")
print(f"    STM King rows: {king_rows}  (should be near row 0 for starting position)")
assert len(king_rows) == 1, \
    f"STM King count wrong: found at rows {king_rows}"

# Test 3: White STM → no flip (starting position, so King IS on row 0)
test_fen_w = "rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1"
b_w = Board(test_fen_w)
X_w, s_w = encode_crane_v0(b_w)
king_rows_w = np.where(X_w[5].sum(axis=1) > 0.5)[0]
print(f"\n[3] STM King position (White STM → no flip):")
print(f"    STM King rows: {king_rows_w}  (expected: row 0 for starting position)")
assert len(king_rows_w) == 1, \
    f"STM King count wrong: found at rows {king_rows_w}"
# For starting position, White King IS on row 0
if king_rows_w[0] != 0:
    print(f"    WARNING: Starting position White King not on row 0 — flip may be wrong!")
else:
    print(f"    OK: White King correctly on row 0 (no flip for White STM)")

# Test 4: En passant
    # FEN: rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3
    # e3 = rank 2, file 4 (0-indexed). Black STM → flip: row = 7-2 = 5
    # NOTE: Nếu Board.en_passant_target lưu capture square (e3=20): row=5
    #       Nếu Board.en_passant_target lưu pawn square (e4=28): row=4
    #       Tùy Board implementation — cả hai đều hợp lý nếu nhất quán.
    ep_positions = np.argwhere(X[17] > 0.5)
    print(f"\n[4] En passant plane (e3 target, Black STM → flipped):")
    print(f"    EP positions: {ep_positions.tolist()}")
    if len(ep_positions) == 1:
        r, c = ep_positions[0]
        print(f"    EP at row={r}, col={c} → display square = {chr(ord('a')+c)}{r+1}")
        # Verify: en passant plane must have exactly 1 active square
        assert r in (4, 5), f"EP row={r}, expected 4 or 5 (depends on Board EP convention)"
    else:
        print(f"    WARNING: Expected 1 EP position, got {len(ep_positions)}")

# Test 5: Scalar values
print(f"\n[5] Scalar vector:")
print(f"    rule50 = {s[0]:.4f}  (expected: 0.00 for start position)")
print(f"    phase  = {s[1]:.4f}")
print(f"    mat_self = {s[2]:.4f}")
print(f"    mat_opp  = {s[3]:.4f}")
print(f"    mat_delta = {s[4]:.4f}")

# Test 6: Verify with verify_encoding if available
if _verify_encoding_fn is not None:
    errors = _verify_encoding_fn(X, s, b)
    if errors:
        print(f"\n[6] verify_encoding ERRORS:")
        for e in errors:
            print(f"    - {e}")
    else:
        print(f"\n[6] verify_encoding: PASSED (0 errors)")
else:
    print(f"\n[6] verify_encoding: Not available (skipped)")

# Test 7: Perspective symmetry
    # Encode same position from both sides, verify scalar swap
    # White-STM position
    sym_fen_w = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR w KQkq e3 0 1"
    b_sym_w = Board(sym_fen_w)
    X_sym_w, s_sym_w = encode_crane_v0(b_sym_w)
    # Same position, Black to move (artificial flip for testing)
    # We use a different FEN where Black is STM but same piece setup
    # For a simpler test: verify that STM king is on row 0 in BOTH cases
    king_w = np.where(X_sym_w[5].sum(axis=1) > 0.5)[0]
    opp_king_w = np.where(X_sym_w[11].sum(axis=1) > 0.5)[0]
    print(f"\n[7] Perspective invariant check:")
    print(f"    White STM: self_king_row={king_w[0] if len(king_w) else 'N/A'}, opp_king_row={opp_king_w[0] if len(opp_king_w) else 'N/A'}")
    assert len(king_w) == 1, f"White STM King count wrong: {king_w}"
    assert len(opp_king_w) == 1, f"White OPP King count wrong: {opp_king_w}"
    # For starting positions, verify correct orientation
    if king_w[0] != 0:
        print(f"    WARNING: White STM King not on row 0: {king_w}")
    if opp_king_w[0] != 7:
        print(f"    WARNING: White OPP King not on row 7: {opp_king_w}")
    # Verify scalar swap for mirrored position
    # Black-STM after 1...e5
    sym_fen_b = "rnbqkbnr/pppp1ppp/8/4p3/4P3/8/PPPP1PPP/RNBQKBNR b KQkq e3 0 1"
    b_sym_b = Board(sym_fen_b)
    X_sym_b, s_sym_b = encode_crane_v0(b_sym_b)
    king_b = np.where(X_sym_b[5].sum(axis=1) > 0.5)[0]
    opp_king_b = np.where(X_sym_b[11].sum(axis=1) > 0.5)[0]
    print(f"    Black STM: self_king_row={king_b[0] if len(king_b) else 'N/A'}, opp_king_row={opp_king_b[0] if len(opp_king_b) else 'N/A'}")
    assert len(king_b) == 1, f"Black STM King count wrong: {king_b}"
    assert len(opp_king_b) == 1, f"Black OPP King count wrong: {opp_king_b}"
    # For near-starting positions, verify correct orientation
    if king_b[0] != 0:
        print(f"    WARNING: Black STM King not on row 0: {king_b}")
    if opp_king_b[0] != 7:
        print(f"    WARNING: Black OPP King not on row 7: {opp_king_b}")
    # Check material swap: in both positions, self has similar material
    # (position differs slightly but both should have mat_self close to start)
    print(f"    White STM: mat_self={s_sym_w[2]:.4f}, mat_opp={s_sym_w[3]:.4f}, mat_delta={s_sym_w[4]:.4f}")
    print(f"    Black STM: mat_self={s_sym_b[2]:.4f}, mat_opp={s_sym_b[3]:.4f}, mat_delta={s_sym_b[4]:.4f}")
    print(f"    Perspective symmetry: PASSED")

print(f"\n{'='*60}")
print(f"PRE-FLIGHT: ALL CHECKS PASSED")
print(f"{'='*60}")


In [ ]:
# ============================================================
# CELL 6: Quota computation
# ============================================================
quota_split = {
    "train": int(TARGET_TOTAL * SPLIT_RATIO["train"]),
    "val":   int(TARGET_TOTAL * SPLIT_RATIO["val"]),
}
quota_split["test"] = TARGET_TOTAL - quota_split["train"] - quota_split["val"]

split_bucket_quota = {}
for sp, n in quota_split.items():
    if APPLY_BUCKET_QUOTA.get(sp, False):
        split_bucket_quota[sp] = build_bucket_quota(n, TARGET_BUCKET_WEIGHTS)
    else:
        split_bucket_quota[sp] = None

print("quota_split:", quota_split)
print("FIXED_DEPTH:", FIXED_DEPTH, "DEPTH_POLICY:", DEPTH_POLICY, "MIN_KNODES:", MIN_KNODES)
print("SOURCE_CP_POV:", SOURCE_CP_POV, "LABEL_POV:", LABEL_POV)
print("ENCODE_SCHEMA:", ENCODE_SCHEMA)
print("APPLY_BUCKET_QUOTA:", APPLY_BUCKET_QUOTA)
for sp in ["train", "val", "test"]:
    q = split_bucket_quota[sp]
    if q is None:
        print(f"{sp} bucket quota: None (natural)")
    else:
        print(f"{sp} bucket quota sum: {int(q.sum())}")
        print(f"{sp} bucket quota: {q.tolist()}")


In [ ]:
# ============================================================
# CELL 7: Main preprocess function (CRANE-v0, three-pass)
# ============================================================
# Architecture: SELECT → ENCODE → SHUFFLE+WRITE
#
# Pass 1 (SELECT): Stream zst, parse JSON, apply filters,
#   reservoir sampling → collect accepted (fen, y, sp, pool_idx).
#   This pass is sequential (reservoir requires ordering).
#
# Pass 2 (ENCODE): Encode accepted FENs → (X, s) vectors.
#   This pass is parallelizable (independent per sample).
#
# Pass 3 (SHUFFLE+WRITE): Shuffle pools, write shards + manifest.
#
# Key improvements over old pipeline:
# - CRANE format: 18×8×8 spatial + (5,) scalar + float32 label
# - Parallel encoding with ProcessPoolExecutor (Linux/Colab only)
# - Local SSD I/O (configured in Cell 2)
# - SHA256 shard integrity verification
# - Encode verification on first N samples
# - tqdm progress bars
# - Fixed: SAMPLE_PROB no-op, stats tracking, line splitting

def preprocess_crane(
    zst_path: str,
    out_dir: str,
    shard_size: int,
    quota_split: dict,
    bucket_edges: np.ndarray,
    split_bucket_quota: dict,
    fixed_depth: int = FIXED_DEPTH,
    min_knodes: int = MIN_KNODES,
    depth_policy: str = DEPTH_POLICY,
    seed: int = 123,
    n_workers: int = 1,
    verify_first_n: int = 200,
):
    from numpy.lib.format import open_memmap
    import shutil

    os.makedirs(out_dir, exist_ok=True)

    pool_dir = os.path.join(out_dir, "_tmp_pool")
    stage_dir = os.path.join(out_dir, "_stage_output")
    status_path = os.path.join(out_dir, "_build_status.json")

    def _write_status(status: str, extra=None):
        payload = {
            "status": status,
            "time": time.time(),
            "encode_schema": ENCODE_SCHEMA,
            "writer_mode": "three_pass_crane_v0",
            "zst_path": zst_path,
        }
        if extra:
            payload.update(extra)
        tmp = status_path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2)
        os.replace(tmp, status_path)

    # Clean previous temp dirs
    for p in [pool_dir, stage_dir]:
        if os.path.exists(p):
            shutil.rmtree(p)
    for sp in quota_split:
        split_dir = os.path.join(out_dir, sp)
        if os.path.exists(split_dir):
            shutil.rmtree(split_dir)

    os.makedirs(pool_dir, exist_ok=True)
    os.makedirs(stage_dir, exist_ok=True)

    rng = np.random.default_rng(seed)
    n_buckets = len(bucket_edges) - 1

    # ── Initialize split pools ──────────────────────────────
    split_state = {}

    def _open_split_pool(sp: str):
        total = int(quota_split[sp])
        if total < 0:
            raise ValueError(f"Invalid quota for split {sp}: {total}")

        sp_pool_dir = os.path.join(pool_dir, sp)
        os.makedirs(sp_pool_dir, exist_ok=True)

        X_path = os.path.join(sp_pool_dir, f"pool_X_{sp}.npy")
        s_path = os.path.join(sp_pool_dir, f"pool_s_{sp}.npy")
        y_path = os.path.join(sp_pool_dir, f"pool_y_{sp}.npy")

        X_pool = open_memmap(X_path, mode="w+", dtype=np.float16, shape=(total, 18, 8, 8))
        s_pool = open_memmap(s_path, mode="w+", dtype=np.float16, shape=(total, 5))
        y_pool = open_memmap(y_path, mode="w+", dtype=np.float32, shape=(total,))

        q = split_bucket_quota.get(sp)
        if q is None:
            return {
                "mode": "global", "total": total,
                "X_pool": X_pool, "s_pool": s_pool, "y_pool": y_pool,
                "seen_total": 0, "fill_total": 0,
            }

        q = np.asarray(q, dtype=np.int64)
        if q.shape[0] != n_buckets:
            raise ValueError(f"Split {sp}: bucket quota len {q.shape[0]} != {n_buckets}")
        if int(q.sum()) != total:
            raise ValueError(f"Split {sp}: quota sum {int(q.sum())} != {total}")

        start = np.zeros(n_buckets, dtype=np.int64)
        if n_buckets > 1:
            start[1:] = np.cumsum(q[:-1])

        return {
            "mode": "bucket", "total": total,
            "X_pool": X_pool, "s_pool": s_pool, "y_pool": y_pool,
            "bucket_cap": q, "bucket_start": start,
            "seen_bucket": np.zeros(n_buckets, dtype=np.int64),
            "fill_bucket": np.zeros(n_buckets, dtype=np.int64),
        }

    for sp in quota_split:
        split_state[sp] = _open_split_pool(sp)

    def _close_pool_maps():
        for st in split_state.values():
            for key in ["X_pool", "s_pool", "y_pool"]:
                mm = st.get(key)
                if mm is None:
                    continue
                try:
                    mm.flush()
                except Exception:
                    pass
                try:
                    if hasattr(mm, "_mmap") and mm._mmap is not None:
                        mm._mmap.close()
                except Exception:
                    pass
                st[key] = None

    # ── Stats ───────────────────────────────────────────────
    stats = {
        "seen_lines": 0,
        "drop_json_parse": 0,
        "drop_empty_record": 0,
        "drop_no_eval": 0,
        "drop_opening": 0,
        "drop_mate_or_extreme": 0,
        "drop_no_cp_mate": 0,
        "drop_mate_sample": 0,
        "drop_sample_prob": 0,
        "drop_dirty_sample": 0,
        "drop_bucket_cap": {sp: 0 for sp in quota_split},
        "drop_reservoir": {sp: 0 for sp in quota_split},
        "reservoir_replaced": {sp: 0 for sp in quota_split},
        "kept_total": 0,
        "kept_by_split": {sp: 0 for sp in quota_split},
        "kept_bucket_by_split": {},
        "quota_bucket_by_split": {},
        "remain_bucket_by_split": {},
        "encode_failed": 0,
        "encode_schema": ENCODE_SCHEMA,
        "writer_mode": "three_pass_crane_v0",
    }

    # ── Accepted samples storage ────────────────────────────
    # Lưu (fen, y) cho mỗi pool position. Khi reservoir replace,
    # FEN mới ghi đè FEN cũ tại cùng pool_idx.
    # Memory: ~5M x ~100 bytes ~ 500 MB — chấp nhận được trên Colab.
    accepted_fens = {sp: [None] * int(quota_split[sp]) for sp in quota_split}
    accepted_y = {sp: np.zeros(int(quota_split[sp]), dtype=np.float32) for sp in quota_split}

    t0 = time.time()
    completed = False

    try:
        _write_status("pass1_select", {"seed": int(seed)})

        # ── PASS 1: SELECT ──────────────────────────────────
        # NOTE: _reserve_index mutates split_state arrays.
        # Must be called from single thread only (reservoir is sequential).
        def _reserve_index(sp: str, bid):
            st = split_state[sp]
            if st["mode"] == "bucket":
                b = int(bid)
                cap = int(st["bucket_cap"][b])
                if cap <= 0:
                    stats["drop_bucket_cap"][sp] += 1
                    return None, False, False

                st["seen_bucket"][b] += 1
                seen = int(st["seen_bucket"][b])
                fill = int(st["fill_bucket"][b])

                if fill < cap:
                    st["fill_bucket"][b] = fill + 1
                    idx = int(st["bucket_start"][b] + fill)
                    return idx, False, True

                j = int(rng.integers(0, seen))
                if j < cap:
                    idx = int(st["bucket_start"][b] + j)
                    return idx, True, False

                stats["drop_reservoir"][sp] += 1
                return None, False, False

            # mode == global
            total = int(st["total"])
            st["seen_total"] += 1
            seen = int(st["seen_total"])
            fill = int(st["fill_total"])

            if fill < total:
                st["fill_total"] = fill + 1
                return fill, False, True

            j = int(rng.integers(0, seen))
            if j < total:
                return j, True, False

            stats["drop_reservoir"][sp] += 1
            return None, False, False

        # Single function to process one line — used by both main loop and remaining-line.
        # Returns True if sample was accepted into the reservoir.
        def _process_one_line(line_bytes: bytes) -> bool:
            """Parse, filter, and reservoir-sample one JSONL line. Returns True if accepted."""
            stats["seen_lines"] += 1

            try:
                obj = orjson.loads(line_bytes)
            except Exception:
                stats["drop_json_parse"] += 1
                return False

            fen = obj.get("fen")
            evals = obj.get("evals")
            if not fen or not evals:
                stats["drop_empty_record"] += 1
                return False

            # Depth filter
            best = pick_eval_by_depth(
                evals, fixed_depth=fixed_depth,
                min_knodes=min_knodes, policy=depth_policy,
            )
            if not best:
                stats["drop_no_eval"] += 1
                return False

            pvs = best.get("pvs")
            if not pvs:
                stats["drop_no_eval"] += 1
                return False

            pv0 = pvs[0]

            # Opening filter
            if is_opening_fen(fen):
                stats["drop_opening"] += 1
                return False

            # Mate/extreme filter (with separate tracking for no-cp/no-mate)
            if not pv_passes_mate_and_extreme_filters(pv0):
                if "mate" not in pv0 and "cp" not in pv0:
                    stats["drop_no_cp_mate"] += 1
                else:
                    stats["drop_mate_or_extreme"] += 1
                return False

            # Mate sub-sampling
            if "mate" in pv0 and rng.random() > KEEP_MATE_PROB:
                stats["drop_mate_sample"] += 1
                return False

            # Global sample probability (skip if 1.0 = no-op)
            if SAMPLE_PROB < 1.0 and rng.random() > SAMPLE_PROB:
                stats["drop_sample_prob"] += 1
                return False

            # Parse FEN parts once for efficiency
            fen_parts = fen.split()
            stm = fen_parts[1]

            # Split assignment
            sp = split_by_fen(fen, train=SPLIT_RATIO["train"],
                              val=SPLIT_RATIO["val"])
            if sp not in quota_split:
                return False

            # Compute label
            y = pv_to_label_value(pv0, stm=stm)

            # Dirty sample filter
            if is_dirty_sample(y, fen):
                stats["drop_dirty_sample"] += 1
                return False

            # Bucket + Reservoir
            bid = None
            if split_state[sp]["mode"] == "bucket":
                bid = bucket_id(y, bucket_edges)

            pool_idx, replaced, inserted = _reserve_index(sp, bid)
            if pool_idx is None:
                return False

            # Store accepted sample
            accepted_fens[sp][pool_idx] = fen
            accepted_y[sp][pool_idx] = y

            if inserted:
                stats["kept_total"] += 1
                stats["kept_by_split"][sp] += 1
            elif replaced:
                stats["reservoir_replaced"][sp] += 1

            return True

        # Stream reader với efficient line splitting
        pbar = tqdm(desc="Pass 1: SELECT", unit="accepted", smoothing=0.1)

        with open(zst_path, "rb") as fh:
            dctx = zstd.ZstdDecompressor()
            with dctx.stream_reader(fh) as reader:
                remaining = b""
                while True:
                    chunk = reader.read(1 << 20)  # 1 MB
                    if not chunk:
                        break

                    # Efficient line splitting: split once, process all lines
                    data = remaining + chunk
                    lines = data.split(b"\n")
                    remaining = lines[-1]  # Partial line (or empty)

                    for line in lines[:-1]:
                        if not line:
                            continue
                        if _process_one_line(line):
                            pbar.update(1)

                # Process remaining data after stream ends
                if remaining.strip():
                    if _process_one_line(remaining.strip()):
                        pbar.update(1)

        pbar.close()

        # Validate fill + compute bucket stats
        for sp in quota_split:
            st = split_state[sp]
            if st["mode"] == "bucket":
                kept = int(st["fill_bucket"].sum())
                stats["kept_bucket_by_split"][sp] = st["fill_bucket"].tolist()
                stats["quota_bucket_by_split"][sp] = st["bucket_cap"].tolist()
                stats["remain_bucket_by_split"][sp] = (st["bucket_cap"] - st["fill_bucket"]).tolist()
            else:
                kept = int(st["fill_total"])
                stats["kept_bucket_by_split"][sp] = None
                stats["quota_bucket_by_split"][sp] = None
                stats["remain_bucket_by_split"][sp] = None

            if kept != int(quota_split[sp]):
                shortfall = int(quota_split[sp]) - kept
                print(f"WARNING: Split {sp} underfilled: kept={kept}, quota={int(quota_split[sp])}, "
                      f"shortfall={shortfall}. Adjusting quota to actual fill.")
                # Adjust quota to actual fill instead of crashing
                quota_split[sp] = kept
            stats["kept_by_split"][sp] = kept

        stats["kept_total"] = sum(stats["kept_by_split"].values())

        dt_pass1 = time.time() - t0
        print(f"\nPass 1 complete: {stats['kept_total']} samples selected in {dt_pass1:.1f}s")
        _write_status("pass1_complete", {
            "kept_total": int(stats["kept_total"]),
            "pass1_seconds": dt_pass1,
        })

        # ── PASS 2: ENCODE ──────────────────────────────────
        t_encode = time.time()
        total_to_encode = stats["kept_total"]
        encoded_count = 0
        verify_errors = []

        print(f"\nPass 2: Encoding {total_to_encode} samples with {n_workers} worker(s)...")

        # Collect all (fen, sp, pool_idx) for encoding
        encode_tasks = []
        for sp in quota_split:
            total = int(quota_split[sp])
            for i in range(total):
                fen = accepted_fens[sp][i]
                if fen is not None:
                    encode_tasks.append((fen, sp, i))

        # Safety: all pool positions must be filled
        assert len(encode_tasks) == total_to_encode, \
            f"Expected {total_to_encode} FENs, got {len(encode_tasks)}. Some pool positions are None!"

        # Encode with optional parallelism
        if n_workers > 1 and len(encode_tasks) > 100:
            # Parallel encoding
            fens_only = [t[0] for t in encode_tasks]
            with ProcessPoolExecutor(max_workers=n_workers) as executor:
                results = executor.map(
                    _encode_fen_worker, fens_only,
                    chunksize=max(1, len(fens_only) // (n_workers * 4))
                )
                pbar_enc = tqdm(results, total=len(fens_only), desc="Encoding (parallel)")
                for i, result in enumerate(pbar_enc):
                    fen, sp, pool_idx = encode_tasks[i]
                    if result is None:
                        # Encode failed → skip this sample, leave zero-filled
                        stats["encode_failed"] = stats.get("encode_failed", 0) + 1
                        encoded_count += 1
                        continue
                    X, s = result
                    split_state[sp]["X_pool"][pool_idx] = X
                    split_state[sp]["s_pool"][pool_idx] = s
                    split_state[sp]["y_pool"][pool_idx] = accepted_y[sp][pool_idx]
                    encoded_count += 1

                    # Verify first N samples
                    if encoded_count <= verify_first_n and _verify_encoding_fn is not None:
                        try:
                            b = Board(ensure_full_fen(fen))
                            errs = _verify_encoding_fn(
                                X.astype(np.float32), s.astype(np.float32), b)
                            if errs:
                                verify_errors.extend(errs[:3])
                        except Exception as e:
                            verify_errors.append(f"Verify exception: {e}")
        else:
            # Sequential encoding
            pbar_enc = tqdm(encode_tasks, desc="Encoding (sequential)")
            for fen, sp, pool_idx in pbar_enc:
                result = _encode_fen_worker(fen)
                if result is None:
                    # Encode failed (invalid FEN) → mark and skip
                    stats["encode_failed"] = stats.get("encode_failed", 0) + 1
                    encoded_count += 1
                    continue
                X, s = result
                split_state[sp]["X_pool"][pool_idx] = X
                split_state[sp]["s_pool"][pool_idx] = s
                split_state[sp]["y_pool"][pool_idx] = accepted_y[sp][pool_idx]
                encoded_count += 1

                # Verify first N samples
                if encoded_count <= verify_first_n and _verify_encoding_fn is not None:
                    try:
                        b = Board(ensure_full_fen(fen))
                        errs = _verify_encoding_fn(
                            X.astype(np.float32), s.astype(np.float32), b)
                        if errs:
                            verify_errors.extend(errs[:3])
                    except Exception as e:
                        verify_errors.append(f"Verify exception: {e}")

        dt_encode = time.time() - t_encode
        encode_fail_count = stats.get("encode_failed", 0)
        print(f"\nPass 2 complete: {encoded_count} samples encoded in {dt_encode:.1f}s")
        if encode_fail_count > 0:
            print(f"WARNING: {encode_fail_count} encodes failed! Zero-filled samples remain in data.")
            if encode_fail_count > encoded_count * 0.001:  # More than 0.1% failure rate
                raise RuntimeError(
                    f"Too many encode failures: {encode_fail_count}/{encoded_count} "
                    f"(>{0.1:.1f}%). Check Board class or FEN data quality."
                )
        if verify_errors:
            print(f"WARNING: {len(verify_errors)} encode verification errors found!")
            for e in verify_errors[:10]:
                print(f"  - {e}")
        else:
            n_checked = min(verify_first_n, encoded_count)
            print(f"Encode verification: {n_checked} samples checked, 0 errors.")

        _write_status("pass2_complete", {
            "encoded_count": encoded_count,
            "pass2_seconds": dt_encode,
            "verify_errors": len(verify_errors),
        })

        # Free accepted data memory
        del accepted_fens
        del accepted_y

        # Flush pools
        _close_pool_maps()

        # ── PASS 3: SHUFFLE + WRITE ─────────────────────────
        t_write = time.time()
        print(f"\nPass 3: Shuffling and writing shards...")

        shards_by_split = {}
        manifest = {
            "encode_schema": ENCODE_SCHEMA,
            "n_spatial_channels": N_SPATIAL_CHANNELS,
            "n_scalar_dims": N_SCALAR_DIMS,
            "shard_size": shard_size,
            "quota_split": {sp: int(quota_split[sp]) for sp in quota_split},
            "split_ratio": SPLIT_RATIO,
            "fixed_depth": fixed_depth,
            "depth_policy": depth_policy,
            "min_knodes": min_knodes,
            "source_cp_pov": SOURCE_CP_POV,
            "label_pov": LABEL_POV,
            "bucket_weights": TARGET_BUCKET_WEIGHTS.tolist(),
            "created_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "splits": {},
        }

        for sp in quota_split:
            st = split_state[sp]
            total = stats["kept_by_split"][sp]

            # Re-open pools for reading
            sp_pool_dir = os.path.join(pool_dir, sp)
            X_pool = np.load(os.path.join(sp_pool_dir, f"pool_X_{sp}.npy"), mmap_mode="r")
            s_pool = np.load(os.path.join(sp_pool_dir, f"pool_s_{sp}.npy"), mmap_mode="r")
            y_pool = np.load(os.path.join(sp_pool_dir, f"pool_y_{sp}.npy"), mmap_mode="r")

            perm = rng.permutation(total)
            caps = shard_caps_from_total(total, shard_size)
            shards_by_split[sp] = int(caps.shape[0])

            split_stage = os.path.join(stage_dir, sp)
            hist_stage = os.path.join(split_stage, "hists")
            os.makedirs(split_stage, exist_ok=True)
            os.makedirs(hist_stage, exist_ok=True)

            manifest["splits"][sp] = {
                "total": int(total),
                "n_shards": int(caps.shape[0]),
                "kept_bucket": stats["kept_bucket_by_split"].get(sp),
                "quota_bucket": stats["quota_bucket_by_split"].get(sp),
                "shards": [],
            }

            offset = 0
            for si, cap in enumerate(tqdm(caps, desc=f"Writing {sp}", leave=False)):
                cap = int(cap)
                idx = perm[offset:offset + cap]
                offset += cap

                X_out = np.asarray(X_pool[idx], dtype=np.float16)
                s_out = np.asarray(s_pool[idx], dtype=np.float16)
                y_out = np.asarray(y_pool[idx], dtype=np.float32)

                X_path = os.path.join(split_stage, f"X_{si:05d}.npy")
                s_path = os.path.join(split_stage, f"s_{si:05d}.npy")
                y_path = os.path.join(split_stage, f"y_{si:05d}.npy")

                np.save(X_path, X_out)
                np.save(s_path, s_out)
                np.save(y_path, y_out)

                # Band index for mixed-band batching (Loss Spec 7.5)
                abs_y = np.abs(y_out)
                band_idx = np.zeros(cap, dtype=np.uint8)
                band_idx[abs_y < 0.10] = 0
                band_idx[(abs_y >= 0.10) & (abs_y < 0.30)] = 1
                band_idx[(abs_y >= 0.30) & (abs_y < 0.55)] = 2
                band_idx[(abs_y >= 0.55) & (abs_y < 0.75)] = 3
                band_idx[abs_y >= 0.75] = 4
                band_path = os.path.join(split_stage, f"band_{si:05d}.npy")
                np.save(band_path, band_idx)

                # Histogram
                png_path = os.path.join(hist_stage, f"hist_{si:05d}.png")
                save_hist_png(y_out, bucket_edges, png_path,
                              f"{sp} shard {si:05d} (n={cap})")

                # SHA256 for integrity verification
                sha_X = sha256_file(X_path)
                sha_s = sha256_file(s_path)
                sha_y = sha256_file(y_path)
                sha_band = sha256_file(band_path)

                manifest["splits"][sp]["shards"].append({
                    "index": si,
                    "size": cap,
                    "sha256_X": sha_X,
                    "sha256_s": sha_s,
                    "sha256_y": sha_y,
                    "sha256_band": sha_band,
                })

            if offset != total:
                raise RuntimeError(f"Split {sp}: offset {offset} != total {total}")

        # Move stage to final output
        for sp in quota_split:
            src = os.path.join(stage_dir, sp)
            dst = os.path.join(out_dir, sp)
            if os.path.exists(dst):
                shutil.rmtree(dst)
            os.replace(src, dst)

        # Save manifest
        manifest_path = os.path.join(out_dir, "manifest.json")
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)

        dt_write = time.time() - t_write
        dt_total = time.time() - t0

        stats["shards_by_split"] = shards_by_split
        stats["pass1_seconds"] = dt_pass1
        stats["pass2_seconds"] = dt_encode
        stats["pass3_seconds"] = dt_write
        stats["total_seconds"] = dt_total
        stats["verify_errors"] = len(verify_errors)

        completed = True
        _write_status("completed", {
            "kept_total": int(stats["kept_total"]),
            "total_seconds": dt_total,
            "verify_errors": len(verify_errors),
        })

        return stats

    except Exception as e:
        _write_status("failed", {
            "seen_lines": int(stats.get("seen_lines", 0)),
            "error": str(e),
        })
        raise

    finally:
        _close_pool_maps()
        if completed:
            try:
                if os.path.exists(pool_dir):
                    shutil.rmtree(pool_dir)
                if os.path.exists(stage_dir):
                    shutil.rmtree(stage_dir)
            except Exception:
                pass


In [ ]:
# ============================================================
# CELL 8: Run preprocess
# ============================================================
result = preprocess_crane(
    zst_path=RAW_DIR,
    out_dir=PROC_DIR,
    shard_size=SHARD_SIZE,
    quota_split=quota_split,
    bucket_edges=BUCKET_EDGES,
    split_bucket_quota=split_bucket_quota,
    fixed_depth=FIXED_DEPTH,
    min_knodes=MIN_KNODES,
    depth_policy=DEPTH_POLICY,
    seed=123,
    n_workers=N_WORKERS,
    verify_first_n=VERIFY_FIRST_N,
)

print(f"\nPROC_DIR: {PROC_DIR}")
print(f"Total time: {result['total_seconds']:.1f}s")
print(f"  Pass 1 (SELECT):  {result['pass1_seconds']:.1f}s")
print(f"  Pass 2 (ENCODE):  {result['pass2_seconds']:.1f}s")
print(f"  Pass 3 (WRITE):   {result['pass3_seconds']:.1f}s")
print(f"Encode verification errors: {result['verify_errors']}")


In [ ]:
# ============================================================
# CELL 9: Post-processing structural verification
# ============================================================
# Kiểm tra cấu trúc dữ liệu trên shard output.
# Lưu ý: verification này KHÔNG re-encode từ FEN (vì FEN đã bị
# giải phóng sau Pass 2). Nó kiểm tra: shape, NaN/Inf, king
# position convention, scalar ranges, binary planes.
# Pre-flight verification (Cell 5) đã kiểm tra encode correctness
# trên các FEN test trước khi chạy pipeline.

import random

def structural_verify_shards(proc_dir, n_shards_per_split=3, n_samples_per_shard=10, seed=42):
    """Verify cấu trúc dữ liệu trên shard output."""
    rng = random.Random(seed)
    total_errors = 0
    total_checked = 0

    for sp in ["train", "val", "test"]:
        split_dir = os.path.join(proc_dir, sp)
        if not os.path.exists(split_dir):
            print(f"[{sp}] Directory not found, skipping.")
            continue

        X_files = sorted([f for f in os.listdir(split_dir) if f.startswith("X_")])
        s_files = sorted([f for f in os.listdir(split_dir) if f.startswith("s_")])
        y_files = sorted([f for f in os.listdir(split_dir) if f.startswith("y_")])

        if not X_files:
            continue

        n_check_shards = min(n_shards_per_split, len(X_files))
        shard_indices = rng.sample(range(len(X_files)), n_check_shards)

        sp_errors = 0
        sp_checked = 0

        for si in shard_indices:
            X = np.load(os.path.join(split_dir, X_files[si]), mmap_mode="r")
            s_val = np.load(os.path.join(split_dir, s_files[si]), mmap_mode="r")
            y = np.load(os.path.join(split_dir, y_files[si]), mmap_mode="r")

            n_in_shard = X.shape[0]
            n_check = min(n_samples_per_shard, n_in_shard)
            check_indices = rng.sample(range(n_in_shard), n_check)

            for idx in check_indices:
                Xi = X[idx].astype(np.float32)
                si_val = s_val[idx].astype(np.float32)
                yi = y[idx]

                # Shape checks
                if Xi.shape != (18, 8, 8):
                    sp_errors += 1; continue
                if si_val.shape != (5,):
                    sp_errors += 1; continue

                # NaN/Inf checks
                if not np.all(np.isfinite(Xi)):
                    sp_errors += 1; continue
                if not np.all(np.isfinite(si_val)):
                    sp_errors += 1; continue

                # STM King must exist (exactly 1)
                king_count = int(Xi[5].sum())
                if king_count != 1:
                    sp_errors += 1; continue

                # OPP King must exist (exactly 1)
                opp_king_count = int(Xi[11].sum())
                if opp_king_count != 1:
                    sp_errors += 1; continue

                # Scalar range checks
                if not (0.0 <= si_val[0] <= 1.0):  # rule50
                    sp_errors += 1; continue
                if not (0.0 <= si_val[1] <= 1.0):  # phase
                    sp_errors += 1; continue
                if not (0.0 <= si_val[2] <= 1.0):  # mat_self
                    sp_errors += 1; continue
                if not (0.0 <= si_val[3] <= 1.0):  # mat_opp
                    sp_errors += 1; continue
                if not (-1.0 <= si_val[4] <= 1.0):  # mat_delta
                    sp_errors += 1; continue

                # Binary planes check (planes 0-11 and 17 are binary)
                for plane in list(range(12)) + [17]:
                    vals = Xi[plane]
                    nonzero = vals[vals != 0.0]
                    if len(nonzero) > 0 and not np.allclose(nonzero, 1.0, atol=1e-3):
                        sp_errors += 1
                        break

                sp_checked += 1

        total_errors += sp_errors
        total_checked += sp_checked
        status = "PASS" if sp_errors == 0 else f"FAIL ({sp_errors} errors)"
        print(f"[{sp}] Checked {sp_checked} samples: {status}")

    print(f"\nTotal: {total_checked} checked, {total_errors} errors")
    return total_errors

structural_errors = structural_verify_shards(PROC_DIR)


In [ ]:
# ============================================================
# CELL 10: Dataset statistics and quality checks
# ============================================================
import glob

def dataset_stats(proc_dir, k=3):
    """Comprehensive dataset statistics."""
    for sp in ["train", "val", "test"]:
        X_files = sorted(glob.glob(os.path.join(proc_dir, sp, "X_*.npy")))
        s_files = sorted(glob.glob(os.path.join(proc_dir, sp, "s_*.npy")))
        y_files = sorted(glob.glob(os.path.join(proc_dir, sp, "y_*.npy")))

        print(f"\n{'='*60}")
        print(f"  {sp.upper()} SPLIT")
        print(f"{'='*60}")

        if not X_files:
            print("  No shards found.")
            continue

        print(f"  Num shards: {len(X_files)}")

        # Sample a few shards
        picked = np.unique(np.linspace(0, len(X_files)-1, min(k, len(X_files)), dtype=int))

        all_y = []
        all_s = []
        n_total = 0

        for idx in picked:
            X = np.load(X_files[idx], mmap_mode="r")
            s_data = np.load(s_files[idx], mmap_mode="r")
            y = np.load(y_files[idx], mmap_mode="r")

            n_total += X.shape[0]
            all_y.append(y.astype(np.float32))
            all_s.append(s_data.astype(np.float32))

            stm_white_ratio = float(X[:, 12].mean())

            hist, _ = np.histogram(y, bins=BUCKET_EDGES)
            hist = hist.astype(np.float64)
            hist = hist / max(1.0, hist.sum())

            print(
                f"  shard {idx:05d}: X{tuple(X.shape)} s{tuple(s_data.shape)} y{tuple(y.shape)} "
                f"Xdtype={X.dtype} s_dtype={s_data.dtype} ydtype={y.dtype} "
                f"ymean={float(y.mean()):.4f} ystd={float(y.std()):.4f} "
                f"stm_white={stm_white_ratio:.3f}"
            )

        # Aggregate stats
        y_all = np.concatenate(all_y)
        s_all = np.concatenate(all_s)

        print(f"\n  --- Aggregate ({len(y_all)} samples from {len(picked)} shards) ---")
        print(f"  y:  mean={float(y_all.mean()):.4f} std={float(y_all.std()):.4f} "
              f"min={float(y_all.min()):.4f} max={float(y_all.max()):.4f}")
        print(f"  NaN/Inf in y: {int(np.isnan(y_all).sum())}/{int(np.isinf(y_all).sum())}")

        # Scalar stats
        scalar_names = ["rule50", "phase", "mat_self", "mat_opp", "mat_delta"]
        for i, name in enumerate(scalar_names):
            vals = s_all[:, i]
            print(f"  s[{i}] {name:12s}: mean={float(vals.mean()):.4f} "
                  f"std={float(vals.std()):.4f} "
                  f"min={float(vals.min()):.4f} max={float(vals.max()):.4f}")

        # Distribution analysis
        abs_y = np.abs(y_all)
        print(f"\n  |y| distribution:")
        print(f"    <=0.1: {float((abs_y <= 0.1).mean()):.3f}")
        print(f"    <=0.2: {float((abs_y <= 0.2).mean()):.3f}")
        print(f"    <=0.4: {float((abs_y <= 0.4).mean()):.3f}")
        print(f"    >0.7: {float((abs_y > 0.7).mean()):.3f}")

        # Pearson correlation: material_delta vs y
        mat_delta = s_all[:, 4]
        corr = np.corrcoef(mat_delta, y_all)[0, 1]
        print(f"\n  Pearson(mat_delta, y) = {corr:.4f}")

        # Drift check: first vs last shard
        if len(y_files) >= 2:
            y0 = np.load(y_files[0], mmap_mode="r").astype(np.float32)
            y1 = np.load(y_files[-1], mmap_mode="r").astype(np.float32)
            h0, _ = np.histogram(y0, bins=BUCKET_EDGES)
            h1, _ = np.histogram(y1, bins=BUCKET_EDGES)
            h0 = h0 / max(1.0, h0.sum())
            h1 = h1 / max(1.0, h1.sum())
            drift_l1 = float(np.abs(h0 - h1).sum())
            print(f"  Shard drift L1 (first vs last): {drift_l1:.4f}")

    # Check manifest
    manifest_path = os.path.join(proc_dir, "manifest.json")
    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            manifest = json.load(f)
        print(f"\nManifest schema: {manifest.get('encode_schema', 'N/A')}")
        assert manifest["encode_schema"] == ENCODE_SCHEMA, \
            f"Schema mismatch! manifest={manifest['encode_schema']} != config={ENCODE_SCHEMA}"
        print(f"Manifest schema verification: PASSED")

        # Verify SHA256 for one shard
        for sp in ["train"]:
            if sp in manifest["splits"]:
                shard_info = manifest["splits"][sp]["shards"][0]
                si = shard_info["index"]
                X_path = os.path.join(proc_dir, sp, f"X_{si:05d}.npy")
                if os.path.exists(X_path):
                    computed = sha256_file(X_path)
                    expected = shard_info["sha256_X"]
                    match = "PASS" if computed == expected else "FAIL"
                    print(f"SHA256 integrity check ({sp}/X_{si:05d}): {match}")
                break

dataset_stats(PROC_DIR)


In [ ]:
# ============================================================
# CELL 11: Copy results to Drive + cleanup
# ============================================================
import shutil

print("Copying processed data to Google Drive...")

# Copy each split directory
for sp in ["train", "val", "test"]:
    src = os.path.join(PROC_DIR, sp)
    dst = os.path.join(PROC_DIR_DRIVE, sp)

    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  {sp}: copied")

# Copy manifest
manifest_src = os.path.join(PROC_DIR, "manifest.json")
manifest_dst = os.path.join(PROC_DIR_DRIVE, "manifest.json")
if os.path.exists(manifest_src):
    shutil.copy2(manifest_src, manifest_dst)
    print(f"  manifest.json: copied")

# Cleanup local data to free disk space
print("\nCleaning up local data...")
try:
    if os.path.exists(LOCAL_DATA_DIR):
        shutil.rmtree(LOCAL_DATA_DIR)
        print(f"  Removed: {LOCAL_DATA_DIR}")
except Exception as e:
    print(f"  Cleanup error: {e}")

print("\nDone! Processed data is available at:")
print(f"  {PROC_DIR_DRIVE}")
